In [1]:
!pip install openai python-dotenv chromadb langchain langchain-community langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

In [2]:
# Install OpenAI SDK if needed
# !pip install -q openai

from openai import OpenAI
from kaggle_secrets import UserSecretsClient

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()

OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")

# OpenRouter client
client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

def get_embedding(text):
    """
    Generate embeddings using OpenRouter
    """

    response = client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    )

    return response.data[0].embedding


# Test
text = "vacation policy"

embedding = get_embedding(text)

print(f"Embedding length: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

Embedding length: 3072
First 5 values: [-0.031982421875, 0.01702880859375, -0.004566192626953125, -0.003391265869140625, 0.03485107421875]


In [3]:
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors

    Returns:
        Float between -1 and 1
    """

    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    dot_product = np.dot(vec1, vec2)

    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)

    # Avoid division by zero
    if norm1 == 0 or norm2 == 0:
        return 0.0

    return dot_product / (norm1 * norm2)


# Test phrases
phrases = [
    "vacation policy",
    "time off rules",
    "PTO guidelines",
    "dress code requirements"
]

# Generate embeddings
embeddings = [get_embedding(p) for p in phrases]

# Base phrase embedding
base = embeddings[0]

print(f'Comparing "{phrases[0]}" with:\n')

# Compare similarities
for i, phrase in enumerate(phrases[1:], start=1):

    similarity = cosine_similarity(base, embeddings[i])

    print(f"{phrase:30} Similarity: {similarity:.4f}")

Comparing "vacation policy" with:

time off rules                 Similarity: 0.6401
PTO guidelines                 Similarity: 0.3733
dress code requirements        Similarity: 0.2564


In [4]:
import os

os.makedirs('company_docs', exist_ok=True)

with open('company_docs/hr_policy.txt', 'w') as f:
    f.write("""Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers.""")

with open('company_docs/benefits.txt', 'w') as f:
    f.write("""Benefits: Health insurance fully covered. 401k matching up to 5%.""")

print("✅ Documents ready!")

✅ Documents ready!


In [5]:
# Install if needed
# !pip install -q chromadb requests

import chromadb
import requests
from kaggle_secrets import UserSecretsClient
from chromadb.api.types import EmbeddingFunction

# ==========================================
# Load OpenRouter API Key from Kaggle Secret
# ==========================================
user_secrets = UserSecretsClient()
OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")

# ==========================================
# OpenRouter Embedding Function
# ==========================================
class OpenRouterEmbeddingFunction(EmbeddingFunction):

    def __init__(
        self,
        api_key,
        model="openai/text-embedding-3-small"
    ):
        self.api_key = api_key
        self.model = model

    def __call__(self, input):

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(
            "https://openrouter.ai/api/v1/embeddings",
            headers=headers,
            json={
                "model": self.model,
                "input": input
            }
        )

        response.raise_for_status()

        data = response.json()

        return [
            item["embedding"]
            for item in data["data"]
        ]


# ==========================================
# Create ChromaDB Client
# ==========================================
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

# ==========================================
# Create Embedding Function
# ==========================================
embedding_fn = OpenRouterEmbeddingFunction(
    api_key=OPENROUTER_API_KEY,
    model="openai/text-embedding-3-small"
)

# ==========================================
# Create / Get Collection
# ==========================================
collection = chroma_client.get_or_create_collection(
    name="company_docs",
    embedding_function=embedding_fn,
    metadata={
        "description": "Company policy documents"
    }
)

# ==========================================
# Verify
# ==========================================
print(f"Collection Name: {collection.name}")
print(f"Document Count: {collection.count()}")

Collection Name: company_docs
Document Count: 0


In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -----------------------------
# Load documents
# -----------------------------
loader = DirectoryLoader(
    "company_docs/",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

# -----------------------------
# Split into chunks
# -----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

# -----------------------------
# Check existing data
# -----------------------------
existing_count = collection.count()

if existing_count == 0:
    ids = []
    texts = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        ids.append(f"doc_{i}_{hash(chunk.page_content)}")  # safer unique ID
        texts.append(chunk.page_content)

        metadatas.append({
            "source": chunk.metadata.get("source", "unknown"),
            "chunk_index": i
        })

    # -----------------------------
    # Add to ChromaDB
    # -----------------------------
    collection.add(
        documents=texts,
        ids=ids,
        metadatas=metadatas
    )

    print(f"✅ Added {len(chunks)} chunks to ChromaDB")

else:
    print(f"⚠️ Collection already contains {existing_count} documents. Skipping ingestion.")

/tmp/ipykernel_16/932578087.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 2 documents
Total chunks created: 2
✅ Added 2 chunks to ChromaDB


In [7]:
def vector_search(query, n_results=3):
    """
    Semantic search using ChromaDB
    """
    return collection.query(
        query_texts=[query],
        n_results=n_results
    )


def print_search_results(query, results):
    """
    Pretty-print search results safely
    """
    print("\n" + "=" * 70)
    print(f"🔎 Query: {query}")
    print("=" * 70)

    documents = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not documents:
        print("No results found.")
        return

    for i, doc in enumerate(documents):
        distance = distances[i] if i < len(distances) else None

        print(f"\nResult {i + 1}")
        if distance is not None:
            print(f"Similarity score (distance): {distance:.4f}")

        print("-" * 50)
        print(doc[:300] + "...")


# -----------------------------
# Test semantic understanding
# -----------------------------
test_queries = [
    "time off policy",   # should match vacation policy
    "WFH guidelines",    # should match remote work policy
    "maternity leave"    # should match parental leave
]

for query in test_queries:
    results = vector_search(query, n_results=2)
    print_search_results(query, results)


🔎 Query: time off policy

Result 1
Similarity score (distance): 0.8960
--------------------------------------------------
Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval.

Parental Leave: 12 weeks pai...

Result 2
Similarity score (distance): 1.4510
--------------------------------------------------
Benefits: Health insurance fully covered. 401k matching up to 5%....

🔎 Query: WFH guidelines

Result 1
Similarity score (distance): 1.5492
--------------------------------------------------
Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval.

Parental Leav

In [8]:
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.5 MB/s eta 0:00:00


In [9]:
# Install if needed
# !pip install -q langchain-openai

from kaggle_secrets import UserSecretsClient
from langchain_openai import ChatOpenAI

# ==========================================
# Load OpenRouter API Key from Kaggle Secret
# ==========================================
user_secrets = UserSecretsClient()

OPENROUTER_API_KEY = user_secrets.get_secret(
    "OPENROUTER_API_KEY"
)

# ==========================================
# LLM Setup (OpenRouter)
# ==========================================
llm = ChatOpenAI(
    model="openai/gpt-3.5-turbo",
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

# ==========================================
# Semantic RAG Function
# ==========================================
def semantic_rag(query, n_results=3):

    # Retrieve relevant documents
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    documents = results.get(
        "documents",
        [[]]
    )[0]

    if not documents:
        return "No relevant information found."

    # Build context
    context = "\n\n---\n\n".join(
        documents[:n_results]
    )

    # Prompt
    messages = [
        (
            "system",
            "You are an HR assistant. "
            "Answer ONLY from the provided context. "
            "If the answer is missing, reply: "
            "'I don't have enough information in the documents.'"
        ),
        (
            "human",
            f"""
Context:
{context}

Question:
{query}

Answer clearly and concisely.
"""
        )
    ]

    # Generate response
    response = llm.invoke(messages)

    return response.content


# ==========================================
# Test Questions
# ==========================================
test_questions = [
    "How much time off do employees get?",
    "Can I work from home?",
    "What is the maternity leave policy?"
]

for question in test_questions:

    print("\n" + "=" * 60)
    print("Question:", question)
    print("=" * 60)

    answer = semantic_rag(question)

    print(answer)


Question: How much time off do employees get?
Employees receive 15 days of paid vacation per year, 12 weeks of paid parental leave for primary caregivers, and may work remotely up to 3 days per week with manager approval.

Question: Can I work from home?
Employees may work remotely up to 3 days per week with manager approval.

Question: What is the maternity leave policy?
12 weeks paid parental leave for primary caregivers.
